# 03 · The tail, and how concentrated it is

**Question.** Does the average describe the passenger experience?

Two statistics answer it: the percentile at which the mean sits, and a Lorenz curve
over positive delay minutes.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


In [2]:
from team01 import tail, viz

panel = clean.completed(io.load_flights())

tail.percentile_profile(panel)

,statistic,value
0,p1,-27.65000
1,p5,-20.81000
2,p10,-16.81000
3,p25,-10.05000
4,p50,-1.68000
5,p75,10.26000
6,p90,53.57000
7,p95,93.19000
8,p99,189.15000
9,p99.9,325.64000


## 3.1 Concentration

The curve is read by position, which is the most accurate encoding available. The
top-1% share is written on the figure because a reader would otherwise estimate it.

In [3]:
concentration = tail.lorenz(panel, sample_rows=2_000_000)
print(f"Gini-style coefficient {concentration['gini']:.3f}")
print(f"worst 1% of flights   -> {concentration['top1_share']:.1%} of delay minutes")
print(f"worst 5% of flights   -> {concentration['top5_share']:.1%} of delay minutes")

viz.fig_lorenz(concentration)

Gini-style coefficient 0.840
worst 1% of flights   -> 15.7% of delay minutes
worst 5% of flights   -> 48.2% of delay minutes


[viz] 05_lorenz.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\05_lorenz.png'

## 3.2 Does the mean hide the tail *between* months?

The pilot found r = 0.998 across months: the mean and the severe share move
together at that level of aggregation. That is a null result, and it is reported
rather than dropped — the "mean hides the tail" claim holds for individual flights,
not between monthly averages.

In [4]:
paradox = tail.mean_vs_tail(panel, by="month")
print(f"Pearson r  {paradox['pearson_r']:+.3f}")
print(f"Spearman r {paradox['spearman_r']:+.3f}")
print(f"verdict    {paradox['verdict']}")

paradox["table"]

Pearson r  +0.574
Spearman r +0.413
verdict    divergence present


,month,mean_delay,share_delayed_60,flights
0,1,7.779992,0.086661,3531
1,2,9.091323,0.094214,1953
2,3,8.079430,0.081304,4723
3,4,9.311350,0.084652,2339
4,5,9.475488,0.085743,5050
5,6,10.569316,0.079351,4625
6,7,13.989498,0.100548,5112
7,8,12.379574,0.092054,3096
8,9,9.067901,0.085706,3512
9,10,9.251082,0.085324,4688


In [5]:
viz.fig_percentile_band(tail.monthly_profile(panel))

[viz] 04_percentile_band.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\04_percentile_band.png'